# Robot path planning by simulation-based policy iteration

A port of `RobotPlanningSimulationBasedPI.mlx`.

The noisy grid of Units 2 and 3, solved by generalized policy iteration from
simulation. The robot crosses a $13\times14$ maze, choosing up, straight or
down at each column; the motion slips, obstacles cost extra, and the walls
push it back.

The driver is the same one the inventory notebook uses, with three
differences that the original scripts have between them -- all exposed as
arguments:

| | inventory script | this script |
|---|---|---|
| terminal Q | set to the cost incurred | set to $0$ |
| behaviour | the current policy $\mu$ | greedy with respect to $Q$ |
| $\epsilon$ | decays after $25\,000$ sims | never decays |

There is also a bug in the original's exploration, which this notebook
measures rather than just describes -- see the end.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

from policy_iteration import sim_based_policy_iteration

rng = np.random.default_rng(0)

In [ ]:
M = np.array([
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1],
    [1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1],
    [1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
n1, n2 = M.shape
c_obst, p_slip = 4, 0.2

def ind2grid(x): return x % n1, x // n1
def grid2ind(i, k): return i + k*n1

In [ ]:
def model_and_cost(x, u):
    """0 = up, 1 = straight, 2 = down. The column always advances."""
    i, k = ind2grid(x)
    if i == 0:                                 # top wall
        i_next, cost = 1, np.sqrt(2) + c_obst
    elif i == n1 - 1:                          # bottom wall
        i_next, cost = n1-2, np.sqrt(2) + c_obst
    elif M[i, k] == 0:                         # obstacle
        i_next, cost = i, c_obst + 1
    else:
        r = rng.random()
        if u == 0:
            i_next, cost = (i+1, np.sqrt(2)) if r > 2*p_slip else (i, 1.0)
        elif u == 1:
            if r < p_slip:        i_next, cost = i+1, np.sqrt(2)
            elif r < 1 - p_slip:  i_next, cost = i,   1.0
            else:                 i_next, cost = i-1, np.sqrt(2)
        else:
            i_next, cost = (i-1, np.sqrt(2)) if r > 2*p_slip else (i, 1.0)
    return grid2ind(i_next, k+1), cost

num_actions = np.array([1 if (M[ind2grid(x)] == 0 or ind2grid(x)[0] in (0, n1-1)) else 3
                        for x in range(n1*n2)])
terminal = list(range(n1*(n2-1), n1*n2))


def exact_stochastic_dp():
    """The same problem with the model, for comparison."""
    s2 = np.sqrt(2); J = np.zeros((n1, n2))
    for k in range(n2-2, -1, -1):
        for i in range(n1):
            if i == 0:          J[i, k] = s2 + c_obst + J[1, k+1]
            elif i == n1-1:     J[i, k] = s2 + c_obst + J[n1-2, k+1]
            elif M[i, k] == 0:  J[i, k] = c_obst + 1 + J[i, k+1]
            else:
                up   = (1-2*p_slip)*(s2+J[i+1, k+1]) + 2*p_slip*(1+J[i, k+1])
                strt = p_slip*(s2+J[i+1, k+1]) + (1-2*p_slip)*(1+J[i, k+1]) \
                       + p_slip*(s2+J[i-1, k+1])
                down = (1-2*p_slip)*(s2+J[i-1, k+1]) + 2*p_slip*(1+J[i, k+1])
                J[i, k] = min(up, strt, down)
    return J

J_exact = exact_stochastic_dp()

In [ ]:
def run(broken_exploration):
    r = np.random.default_rng(0)
    global rng; rng = r
    Q, mu, J, hist = sim_based_policy_iteration(
        num_actions, model_and_cost,
        nsim=30_000, eps=0.1, alpha=0.2,
        prob_init_state=np.r_[np.ones(n1)/n1, np.zeros(n1*(n2-1))],
        terminal_states=terminal, outer=10, rng=r,
        terminal_cost=False, greedy_behaviour=True, eps_decay_at=None,
        broken_exploration=broken_exploration, verbose=False)
    return np.array([[J[grid2ind(i, k)] for k in range(n2)] for i in range(n1)])

J_fixed = run(broken_exploration=False)
print("learned cost-to-go, first column:", np.round(J_fixed[:, 0], 2))
print("exact,              first column:", np.round(J_exact[:, 0], 2))
print(f"\nmax error over the grid: {np.abs(J_fixed - J_exact)[:, :-1].max():.2f}")
print(f"best starting row -- exact {1+np.argmin(J_exact[1:n1-1, 0])}, "
      f"learned {1+np.argmin(J_fixed[1:n1-1, 0])}")

## A bug in the original's exploration

The script explores with

```matlab
u(k) = sampledist(ones(1,nu(x(k))));
```

an **unnormalised** vector of ones. Inside `sampledist` the cumulative sum
is then `[1,2,3]`, and the draw is `a = rand(1)`, which is always below $1$.
So `sum(cprobdist_ < a)` is always $0$ and the "random" exploratory action is
always the **first** one -- up. Exploration is not uniform over the three
actions; it is "always try up".

The inventory script does this correctly, with `ones(1,nu(x(k)))/nu(x(k))`.
The fix is one `/nu`.

The cell below runs both and measures what the bug costs.

In [ ]:
J_broken = run(broken_exploration=True)

print(f"{'':22s}{'max err, first column':>24s}{'max err, grid':>16s}")
for name, Jg in (("as written", J_broken), ("exploration fixed", J_fixed)):
    print(f"{name:22s}{np.abs(Jg[:,0]-J_exact[:,0]).max():>24.2f}"
          f"{np.abs(Jg-J_exact)[:,:-1].max():>16.2f}")

fig, ax = plt.subplots(figsize=(9, 4))
rows = np.arange(n1)
ax.plot(rows, J_exact[:, 0],  "k:",  label="exact")
ax.plot(rows, J_fixed[:, 0],  "o-",  label="exploration fixed")
ax.plot(rows, J_broken[:, 0], "s--", label="as written (always explores 'up')")
ax.set_xlabel("starting row"); ax.set_ylabel("cost-to-go from the first column")
ax.grid(True); ax.legend()
plt.tight_layout(); plt.show()

Fixing the one character more than halves the error. With the bug the
middle rows are overestimated by about six -- the robot never discovers that
going *down* is sometimes right, because the only action exploration ever
offers it is up.

It is a good illustration of the unit's point that exploration is the
recurring difficulty in simulation-based policy iteration: here the
algorithm, the step size and the number of simulations are all fine, and the
answer is still wrong because one branch of the action space was never
tried.